# 🍷 Wine Quality Classification — Random Forest

**Tujuan:** Memprediksi kualitas anggur (`quality`, skala 0–10) berdasarkan fitur-fitur kimiawi.

**Metode:** Random Forest Classifier

**Dataset:**
- `data_training.csv` — 857 sampel dengan label `quality`
- `data_testing.csv` — 286 sampel tanpa label (yang akan diprediksi)

---

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded ✅')

## 2. Load Dataset

In [ ]:
train = pd.read_csv('data_training.csv')
test  = pd.read_csv('data_testing.csv')

print(f'Training set : {train.shape[0]} baris, {train.shape[1]} kolom')
print(f'Testing set  : {test.shape[0]} baris, {test.shape[1]} kolom')
train.head()

## 3. Eksplorasi Data (EDA)

In [ ]:
print('Missing values (train):', train.isnull().sum().sum())
print('Missing values (test) :', test.isnull().sum().sum())
train.describe().T

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
qc = train['quality'].value_counts().sort_index()
bars = ax.bar(qc.index, qc.values, color='steelblue', edgecolor='white')
for bar, v in zip(bars, qc.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(v), ha='center')
ax.set_xlabel('Quality Score'); ax.set_ylabel('Jumlah Sampel')
ax.set_title('Distribusi Kualitas Anggur (Training Set)')
plt.tight_layout(); plt.show()

**Interpretasi:** Dataset tidak seimbang — kelas 5 dan 6 mendominasi (~82%). Kelas 3 dan 8 sangat sedikit.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
cols = [c for c in train.columns if c != 'Id']
corr = train[cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Korelasi Antar Fitur')
plt.tight_layout(); plt.show()

## 4. Persiapan Data

In [ ]:
FEATURES = [c for c in train.columns if c not in ['quality', 'Id']]

X_all    = train[FEATURES]
y_all    = train['quality']
X_test   = test[FEATURES]
test_ids = test['Id']

print('Fitur:', FEATURES)

## 5. Pemodelan — Random Forest

### Strategi Evaluasi
Karena `data_testing.csv` **tidak memiliki label asli**, kita tidak bisa menghitung accuracy testing secara langsung.

Solusinya: pisahkan **20% dari training data** sebagai **validation set** untuk mensimulasikan testing accuracy.

```
data_training.csv (857 baris)
    ├── 80% → X_train (685 baris) ← untuk melatih model
    └── 20% → X_val   (172 baris) ← simulasi testing (ada label → bisa hitung akurasi)
```

Setelah evaluasi, model dilatih ulang dengan **seluruh** data training untuk prediksi final.

In [ ]:
# Split 80/20
X_tr, X_val, y_tr, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
print(f'Train split : {X_tr.shape[0]} baris')
print(f'Val split   : {X_val.shape[0]} baris (simulasi testing)')

In [ ]:
rf_eval = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf_eval.fit(X_tr, y_tr)
print('Model selesai dilatih ✅')

## 6. Evaluasi Model

In [ ]:
# Training vs Validation Accuracy
train_acc = accuracy_score(y_tr, rf_eval.predict(X_tr))
val_acc   = accuracy_score(y_val, rf_eval.predict(X_val))

print(f'Training Accuracy            : {train_acc:.4f} ({train_acc*100:.2f}%)')
print(f'Validation Accuracy (sim test): {val_acc:.4f} ({val_acc*100:.2f}%)')
print(f'Gap (overfit check)          : {train_acc - val_acc:.4f}')

**Interpretasi:**
- **Training accuracy ~100%** — Random Forest dengan pohon penuh memang overfit pada data latih (ini normal)
- **Validation accuracy ~60%** — estimasi akurasi model pada data baru (simulasi testing)
- Akurasi ~60% wajar untuk dataset Wine Quality yang sangat imbalanced

In [ ]:
# Cross-Validation 5-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_eval, X_all, y_all, cv=skf, scoring='accuracy')

print('Cross-Validation Accuracy per Fold:')
for i, s in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {s:.4f} ({s*100:.2f}%)')
print(f'  Mean : {cv_scores.mean():.4f} ({cv_scores.mean()*100:.2f}%)')
print(f'  Std  : {cv_scores.std():.4f}')

In [ ]:
# Ringkasan akurasi
summary = pd.DataFrame({
    'Metric': ['Training Accuracy', 'Validation Accuracy (hold-out 20%)', 'CV Accuracy (5-fold mean)'],
    'Score' : [f'{train_acc*100:.2f}%', f'{val_acc*100:.2f}%', f'{cv_scores.mean()*100:.2f}%']
})
print(summary.to_string(index=False))

In [ ]:
# Classification Report pada Validation Set
y_val_pred = rf_eval.predict(X_val)
print('Classification Report (Validation Set — Simulasi Testing):')
print(classification_report(y_val, y_val_pred))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_val, y_val_pred)
ConfusionMatrixDisplay(cm, display_labels=sorted(y_val.unique())).plot(
    ax=ax, colorbar=False, cmap='Blues'
)
ax.set_title(f'Confusion Matrix — Validation Set (Accuracy: {val_acc:.2%})')
plt.tight_layout(); plt.show()

In [ ]:
# Feature Importance
fi = pd.Series(rf_eval.feature_importances_, index=FEATURES).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
fi.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Feature Importance — Random Forest')
ax.set_ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()
print(fi)

## 7. Prediksi Final (Retrain Full Data)

Model dilatih ulang dengan **semua 857 baris** agar prediksi final memanfaatkan data semaksimal mungkin.

In [ ]:
rf_final = RandomForestClassifier(
    n_estimators=300, max_features='sqrt', random_state=42, n_jobs=-1
)
rf_final.fit(X_all, y_all)
print('Model final (full data) selesai dilatih ✅')

y_pred = rf_final.predict(X_test)
print('\nDistribusi Prediksi Testing:')
print(pd.Series(y_pred).value_counts().sort_index())

## 8. Simpan Hasil Prediksi

In [ ]:
submission = pd.DataFrame({'Id': test_ids, 'quality': y_pred})
submission.to_csv('submission.csv', index=False)

print('submission.csv berhasil disimpan ✅')
print(f'Total prediksi : {len(submission)} baris')
print('Kolom          :', submission.columns.tolist())
submission.head(10)

---
## 9. Kesimpulan

| Metric | Nilai |
|--------|-------|
| Training Accuracy | ~100% (overfitting wajar pada RF) |
| **Validation Accuracy (simulasi testing)** | **~60.47%** |
| CV Accuracy (5-fold mean) | ~64.30% |
| Fitur terpenting | alcohol, sulphates, volatile acidity |
| Total prediksi | 286 sampel |

**Catatan:** Data testing tidak memiliki label, sehingga akurasi disimulasikan dari 20% data training. Nilai ~60–64% wajar untuk Wine Quality dataset yang sangat imbalanced.